In [1]:
# import os
# import shutil
# import site
# import sys

# def setup_coqui_env():
#     print("🧹 مرحله ۱: پاکسازی عمیق تداخل‌های پایتون ۳.۱۲...")
    
#     # حذف نصب‌های قبلی
#     !pip uninstall -y -q coqpit coqpit-config coqui-tts TTS coqui-tts-trainer monotonic-alignment-search
    
#     # حذف فیزیکی پوشه‌ها
#     paths = site.getsitepackages()
#     for path in paths:
#         for folder in ['coqpit', 'coqpit_config', 'TTS', 'coqui_tts', 'trainer', 'monotonic_alignment_search']:
#             target = os.path.join(path, folder)
#             if os.path.exists(target):
#                 shutil.rmtree(target)
    
#     print("📥 مرحله ۲: نصب موتور محاسباتی (این بخش برای VITS حیاتی است)...")
#     # نصب مستقیم از سورس برای اطمینان از کامپایل درست در کگل
#     !pip install -q --no-cache-dir monotonic-alignment-search
    
#     print("📥 مرحله ۳: نصب کتابخانه‌های صوتی و هماهنگ‌سازی نسخه‌ها...")
#     # نصب پیشنیازها با نسخه‌های تست شده
#     !pip install -q --no-cache-dir "numpy<2.0" "scipy<1.13" "pandas<2.1.0"
#     !pip install -q --no-cache-dir "transformers<5.0.0" "pysbd" "encodec" "gruut"
    
#     # نصب TTS و Trainer
#     !pip install -q --no-cache-dir "coqui-tts-trainer>=0.3.0"
#     !pip install -q --no-cache-dir "coqui-tts"
    
#     # نصب ابزار سیستمی
#     !apt-get install -y -qq espeak-ng > /dev/null
    
#     # اضافه کردن مسیرهای نصب شده به پایتون فعلی
#     for path in site.getsitepackages():
#         if path not in sys.path:
#             sys.path.append(path)

#     print("✅ مرحله ۴: بررسی نهایی نصب...")
#     try:
#         import monotonic_alignment_search
#         print("⚡ موتور monotonic_alignment_search با موفقیت شناسایی شد.")
#     except ImportError:
#         print("⚠️ موتور شناسایی نشد، اما بعد از Restart Session درست می‌شود.")

#     print("\n" + "="*50)
#     print("🚀 نصب تمام شد! حالا حتماً از منوی بالا 'Run > Restart Session' را بزنید.")
#     print("="*50)

# # اجرای نصب
# setup_coqui_env()

In [3]:
try:
    import TTS
    from trainer import Trainer
    import numpy as np
    print(f"✅ Numpy Version: {np.__version__}") # باید زیر 2.0.0 باشد
    print("🚀 محیط آماده است! برو برای آموزش ۱۰۰ اپوک.")
except Exception as e:
    print(f"❌ هنوز یک مشکل کوچک هست: {e}")

2026-02-16 10:56:53.907492: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1771239414.091219     273 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1771239414.146770     273 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1771239414.626976     273 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771239414.627016     273 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771239414.627018     273 computation_placer.cc:177] computation placer alr

✅ Numpy Version: 2.0.2
🚀 محیط آماده است! برو برای آموزش ۱۰۰ اپوک.


In [4]:
import transformers
print(f"Transformers version: {transformers.__version__}")
import TTS
print("TTS is ready!")

Transformers version: 4.57.1
TTS is ready!


In [5]:
import torch
from trainer import Trainer, TrainerArgs

# ۱. چک کردن نسخه و صحت نصب
try:
    import trainer
    print(f"✅ Trainer Version: {trainer.__version__}")
    
    # ۲. تست یک عملیات بسیار ساده با TrainerArgs
    args = TrainerArgs()
    print("✅ TrainerArgs initialized successfully!")
    
    # ۳. چک کردن دسترسی به GPU (حیاتی برای سرعت ۱۰۰ اپوک)
    if torch.cuda.is_available():
        print(f"🚀 GPU is READY: {torch.cuda.get_device_name(0)}")
    else:
        print("⚠️ GPU is NOT available. Training will be VERY slow!")

except Exception as e:
    print(f"❌ مشکلی در نصب وجود دارد: {e}")

✅ Trainer Version: 0.3.3
✅ TrainerArgs initialized successfully!
🚀 GPU is READY: Tesla P100-PCIE-16GB


In [ ]:
import os
import sys
import torch
from types import ModuleType

# ۱. تنظیمات GPU و پچ زبان
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# پچ کردن دستی برای جلوگیری از خطای ko_speech
m = ModuleType("ko_speech_tools")
m.hangul_romanize = lambda x: x
sys.modules["ko_speech_tools"] = m

# ۲. ایمپورت‌های اصلی
from trainer import Trainer, TrainerArgs
from TTS.tts.configs.shared_configs import CharactersConfig
from TTS.tts.configs.vits_config import VitsConfig
from TTS.tts.models.vits import Vits
from TTS.utils.audio import AudioProcessor
from TTS.tts.utils.text.tokenizer import TTSTokenizer

# ۳. مسیرهای دیتاسِت (مطمئن شو این مسیرها دقیق هستند)
WAV_DIR = "/kaggle/input/datasets/magnoliasis/persian-tts-dataset-famale/wavs"
TRAIN_TXT = "/kaggle/input/datasets/morvaridrhbr/textual-data/Data/train.txt"
VAL_TXT = "/kaggle/input/datasets/morvaridrhbr/textual-data/Data/val.txt"

def load_persian_samples(meta_file):
    items = []
    if not os.path.exists(meta_file): return []
    with open(meta_file, "r", encoding="utf-8") as f:
        for line in f:
            if "|" in line:
                parts = line.strip().split("|")
                wav_name = os.path.basename(parts[0].strip())
                full_wav_path = os.path.join(WAV_DIR, wav_name)
                if os.path.exists(full_wav_path):
                    items.append({"text": parts[1].strip(), "audio_file": full_wav_path, "speaker_name": "female", "language": "fa", "audio_unique_name": wav_name})
    return items

train_samples = load_persian_samples(TRAIN_TXT)
eval_samples = load_persian_samples(VAL_TXT)

# ۴. تنظیمات بهینه برای جلوگیری از ارور
config = VitsConfig(
    run_name="vits_persian_v1",
    batch_size=32,
    eval_batch_size=8,
    num_loader_workers=2, # تعداد کمتر برای پایداری در کگل
    text_cleaner="basic_cleaners",
    use_phonemes=False,
    characters=CharactersConfig(
        characters="ABCDEGHJKLMNOPQRSTUVWXZabcdefghijklmnopqrstuvwxyz? ",
        punctuations="!(),.-:;",
        pad="<pad>", eos="<eos>", bos="<sos>", blank="@"
    ),
    output_path="/kaggle/working/tts_results",
    epochs=100,
    mixed_precision=True
)

# ۵. اجرای نهایی با هندل کردن موتور محاسباتی
try:
    if len(train_samples) == 0:
        print("🛑 فایل صوتی پیدا نشد!")
    else:
        print(f"🔥 آماده آموزش {len(train_samples)} فایل...")
        
        # --- بخش اصلاح شده برای شناسایی موتور ---
        import monotonic_alignment_search
        # تزریق دستی تابع به مدل VITS اگر پایتون پیدایش نکرد
        try:
            from monotonic_alignment_search import maximum_path
        except ImportError:
            from monotonic_alignment_search.monotonic_alignment_search import maximum_path
        
        torch.backends.cudnn.benchmark = True
        ap = AudioProcessor.init_from_config(config)
        tokenizer, _ = TTSTokenizer.init_from_config(config)
        model = Vits(config, ap, tokenizer)

        trainer = Trainer(
            TrainerArgs(), 
            config, 
            output_path=config.output_path, 
            model=model, 
            train_samples=train_samples, 
            eval_samples=eval_samples
        )
        trainer.fit()
except Exception as e:
    print(f"❌ خطای نهایی: {str(e)}")